# Telecom Customer Usage Analysis

## Project Overview

This project analyzes telecom customer data to understand:

- Customer data usage patterns
- High, medium, and low usage customers
- Application usage behavior
- Handset manufacturer and handset type preferences
- Network performance
- Download and upload behavior

## Tools Used

- MySQL
- MySQL Workbench
- Python
- Jupyter Notebook
- Power BI
- GitHub


# 1. Dataset Information

The dataset contains telecom customer information including:

- Customer/MSISDN number
- Handset type and manufacturer
- Total download and upload data
- Application usage such as YouTube, Netflix, Gaming, Google, Email, and Social Media
- Download and upload speeds
- Network latency (RTT)
- TCP retransmission information

The dataset was imported into MySQL for SQL-based exploratory analysis.

**Database:** `telecom_project`  
**Table:** `telecom_data_sample`

# 2. SQL Data Exploration

Before performing detailed customer analysis, the dataset was explored to understand its structure, size, and customer coverage.

## 2.1 Preview the Dataset

```sql
SELECT *
FROM telecom_data_sample
LIMIT 10;
```

## 2.2 Total Records and Unique Customers

The following query checks the total number of records and the number of unique customers in the dataset.

```sql
SELECT
    COUNT(*) AS total_records,
    COUNT(DISTINCT `MSISDN/Number`) AS unique_customers
FROM telecom_data_sample;
```

## 2.3 Check Missing Customer IDs

This query checks whether the customer identifier (`MSISDN/Number`) contains any missing values.

```sql
SELECT 
    COUNT(*) AS missing_customer_ids
FROM telecom_data_sample
WHERE `MSISDN/Number` IS NULL;
```

## 2.4 Top Handset Manufacturers

This query identifies the most commonly used handset manufacturers among customers.

```sql
SELECT
    `Handset Manufacturer`,
    COUNT(*) AS customer_count
FROM telecom_data_sample
WHERE `Handset Manufacturer` IS NOT NULL
    AND `Handset Manufacturer` <> 'undefined'
GROUP BY `Handset Manufacturer`
ORDER BY customer_count DESC
LIMIT 10;
```

## 2.5 Top Handset Types

This query identifies the most commonly used handset models among customers.

```sql
SELECT
    `Handset Type`,
    COUNT(*) AS customer_count
FROM telecom_data_sample
WHERE `Handset Type` IS NOT NULL
    AND `Handset Type` <> 'undefined'
GROUP BY `Handset Type`
ORDER BY customer_count DESC
LIMIT 10;
```

## 3. Customer Data Usage Analysis

This section analyzes the total download and upload data consumed by customers.

### 3.1 Total Data Usage by Customer

The following query calculates the total data usage for each customer by combining download and upload traffic and converting the result from bytes to GB.

```sql
SELECT
    `MSISDN/Number` AS customer_id,
    ROUND(
        SUM(`Total DL (Bytes)` + `Total UL (Bytes)`) / 1000000000,
        2
    ) AS total_data_gb
FROM telecom_data_sample
WHERE `MSISDN/Number` IS NOT NULL
GROUP BY `MSISDN/Number`
ORDER BY total_data_gb DESC
LIMIT 10;
```

### 3.2 Customer Usage Segmentation

Customers are classified into High, Medium, and Low Usage categories based on their total download and upload data consumption.

```sql
SELECT
    CASE
        WHEN (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000
            THEN 'High Usage'
        WHEN (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 400000000
            THEN 'Medium Usage'
        ELSE 'Low Usage'
    END AS usage_category,
    COUNT(*) AS customer_count
FROM telecom_data_sample
WHERE `MSISDN/Number` IS NOT NULL
GROUP BY usage_category
ORDER BY customer_count DESC;
```

### 3.3 Average Download and Upload Speed by Usage Category

This analysis compares the average download and upload speeds across High, Medium, and Low Usage customers.

```sql
SELECT
    CASE
        WHEN (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000
            THEN 'High Usage'
        WHEN (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 400000000
            THEN 'Medium Usage'
        ELSE 'Low Usage'
    END AS usage_category,

    ROUND(AVG(`Avg RTT DL (ms)`), 2) AS avg_rtt_download_ms,
    ROUND(AVG(`Avg RTT UL (ms)`), 2) AS avg_rtt_upload_ms

FROM telecom_data_sample

WHERE `MSISDN/Number` IS NOT NULL

GROUP BY usage_category

ORDER BY avg_rtt_download_ms ASC;
```

### 3.4 TCP Retransmission by Usage Category

This analysis compares the average TCP retransmission volume across High, Medium, and Low Usage customers. Lower retransmission generally indicates fewer data packets needing to be resent.

```sql
SELECT
    CASE
        WHEN (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000
            THEN 'High Usage'
        WHEN (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 400000000
            THEN 'Medium Usage'
        ELSE 'Low Usage'
    END AS usage_category,

    ROUND(
        AVG(`TCP DL Retrans. Vol (Bytes)`) / 1000000,
        2
    ) AS avg_tcp_dl_retrans_mb,

    ROUND(
        AVG(`TCP UL Retrans. Vol (Bytes)`) / 1000000,
        2
    ) AS avg_tcp_ul_retrans_mb

FROM telecom_data_sample
WHERE `MSISDN/Number` IS NOT NULL
GROUP BY usage_category
ORDER BY avg_tcp_dl_retrans_mb ASC;
```

## 4. Application Usage Analysis

This section analyzes data consumption across major applications to identify which services generate the highest network traffic.

### 4.1 Total Application Data Usage

The following query compares total data usage across Gaming, YouTube, Netflix, Google, Email, and Social Media.

```sql
SELECT 'Gaming' AS application,
       ROUND(SUM(`Gaming DL (Bytes)` + `Gaming UL (Bytes)`) / 1000000000, 2) AS usage_gb
FROM telecom_data_sample

UNION ALL

SELECT 'YouTube',
       ROUND(SUM(`Youtube DL (Bytes)` + `Youtube UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample

UNION ALL

SELECT 'Netflix',
       ROUND(SUM(`Netflix DL (Bytes)` + `Netflix UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample

UNION ALL

SELECT 'Google',
       ROUND(SUM(`Google DL (Bytes)` + `Google UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample

UNION ALL

SELECT 'Email',
       ROUND(SUM(`Email DL (Bytes)` + `Email UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample

UNION ALL

SELECT 'Social Media',
       ROUND(SUM(`Social Media DL (Bytes)` + `Social Media UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample

ORDER BY usage_gb DESC;
```

### 4.2 Application Usage for High Usage Customers

This analysis examines application data consumption specifically among high-usage customers.

```sql
SELECT 'Gaming' AS application,
       ROUND(SUM(`Gaming DL (Bytes)` + `Gaming UL (Bytes)`) / 1000000000, 2) AS usage_gb
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000

UNION ALL

SELECT 'YouTube',
       ROUND(SUM(`Youtube DL (Bytes)` + `Youtube UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000

UNION ALL

SELECT 'Netflix',
       ROUND(SUM(`Netflix DL (Bytes)` + `Netflix UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000

UNION ALL

SELECT 'Google',
       ROUND(SUM(`Google DL (Bytes)` + `Google UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000

UNION ALL

SELECT 'Email',
       ROUND(SUM(`Email DL (Bytes)` + `Email UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000

UNION ALL

SELECT 'Social Media',
       ROUND(SUM(`Social Media DL (Bytes)` + `Social Media UL (Bytes)`) / 1000000000, 2)
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000

ORDER BY usage_gb DESC;
```

## 5. High Usage Customer Analysis

This section focuses on customers with high data consumption to understand their handset preferences and identify the highest-consuming customers.

### 5.1 Top Handset Models Among High Usage Customers

The following query identifies the most commonly used handset models among high-usage customers.

```sql
SELECT
    `Handset Type`,
    COUNT(*) AS customer_records
FROM telecom_data_sample
WHERE `MSISDN/Number` IS NOT NULL
    AND `Handset Type` IS NOT NULL
    AND `Handset Type` <> 'undefined'
    AND (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000
GROUP BY `Handset Type`
ORDER BY customer_records DESC
LIMIT 10;
```

### 5.2 Top Handset Manufacturers Among High Usage Customers

This analysis identifies the handset manufacturers most commonly used by high-usage customers.

```sql
SELECT
    `Handset Manufacturer`,
    COUNT(DISTINCT `MSISDN/Number`) AS customer_count
FROM telecom_data_sample
WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000
    AND `MSISDN/Number` IS NOT NULL
    AND `Handset Manufacturer` IS NOT NULL
    AND `Handset Manufacturer` <> 'undefined'
GROUP BY `Handset Manufacturer`
ORDER BY customer_count DESC
LIMIT 10;
```

### 5.3 Top 10 High Usage Customers

This analysis identifies the customers with the highest total data consumption within the High Usage segment.

```sql
SELECT
    `MSISDN/Number` AS customer_id,

    ROUND(
        SUM(`Total DL (Bytes)` + `Total UL (Bytes)`) / 1000000000,
        2
    ) AS total_data_gb

FROM telecom_data_sample

WHERE (`Total DL (Bytes)` + `Total UL (Bytes)`) >= 700000000
    AND `MSISDN/Number` IS NOT NULL

GROUP BY `MSISDN/Number`

ORDER BY total_data_gb DESC

LIMIT 10;
```

## 6. Key Business Insights

Based on the SQL analysis, the following key insights were identified:

1. **Customer Base:** The dataset contains 9,741 records representing 8,235 unique customers.

2. **Data Consumption:** Average data usage per customer is approximately 0.58 GB.

3. **Application Usage:** Gaming generated the highest data traffic at approximately 4,173.45 GB, followed by YouTube (219.17 GB) and Netflix (218.88 GB).

4. **High Usage Customers:** Gaming also dominated application traffic within the High Usage segment, generating approximately 1,853.24 GB.

5. **Handset Manufacturers:** Apple had the largest number of high-usage customers, followed by Huawei and Samsung.

6. **Handset Models:** Huawei B528S-23A was the most common individual handset model among high-usage records.

7. **Network Latency:** High Usage records had the lowest average download RTT at approximately 110.64 ms, compared with Low Usage at 117.10 ms and Medium Usage at 137.13 ms.

8. **TCP Retransmission:** High Usage records had the lowest average download TCP retransmission among the three usage categories at approximately 9.79 MB.

9. **Download vs Upload:** Customer data consumption was primarily download-driven, with download usage substantially higher than upload usage.

## 7. Conclusion

The SQL analysis provided insights into customer data consumption, application usage, handset preferences, and network performance.

The analysis showed that gaming generated the highest amount of application traffic, while YouTube and Netflix also contributed significant data usage. Apple had the largest number of high-usage customers, although Huawei B528S-23A was the most common individual handset model among high-usage records.

Network performance analysis using RTT and TCP retransmission provided additional insight into the experience of different usage groups.

These findings can help telecom businesses better understand customer behavior, network demand, and high-usage customer segments. The insights can also support targeted data plans, network optimization, and customer-focused business strategies.